# 06 · Reading a training run

Companion notebook for [Chapter 7](../chapters/ch07-reading-runs.md).

**This notebook needs no GPU and no model.** It downloads the four training logs
that ship with the course and teaches you to diagnose a run from its metrics.

**You will:** compare four GRPO variants, find the collapse, and prove the
collapsed model is outputting a uniform distribution.

In [ ]:
import io, urllib.request
import pandas as pd

BASE = ("https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/"
        "main/ch07/02_logs/")

FILES = {
    "ch6 original (no KL)": "ch06_rlvr_grpo_original_no_kl_metrics.csv",
    "+ tracking":           "7_3_plus_tracking_metrics.csv",
    "+ clipped ratio":      "7_4_plus_clip_ratio_metrics.csv",
    "+ KL term":            "7_5_plus_kl_metrics.csv",
    "+ format reward":      "7_6_plus_format_reward_metrics.csv",
}

logs = {}
for label, fname in FILES.items():
    with urllib.request.urlopen(BASE + fname, timeout=60) as r:
        logs[label] = pd.read_csv(io.StringIO(r.read().decode()))
    print(f"{label:24} {len(logs[label]):>4} steps, columns: {list(logs[label].columns)}")

## 1 · The metric dictionary

| Column | Healthy | Alarm |
| --- | --- | --- |
| `reward_avg` | rises, plateaus | 0 = no signal; 1 = too easy |
| `policy_ratio` | near 1.0 | 3×, 10× = moved too far |
| `kl_loss` | small, stable | equals `loss` = KL took over |
| `entropy_avg` | slow decline | 0 = no exploration; ~11.9 = uniform |
| `adv_std` | non-zero | 0 = no learning signal |
| `avg_response_len` | gradual | runaway = reward hacking |
| `eval_acc` | rises then plateaus | falls while reward rises = hacking |

## 2 · Accuracy across variants

In [ ]:
for label, df in logs.items():
    if "eval_acc" not in df:
        continue
    ev = df.dropna(subset=["eval_acc"])[["step", "eval_acc"]]
    print(f"\n=== {label} ===")
    print(ev.to_string(index=False))

**What you should see:** four of five runs start at 15.6% (the full-test base
accuracy). 7_6 starts at 50.8%, so its curve is not directly comparable — a good
reminder to check starting points before comparing runs.

The original and "+ tracking" rows should be **identical**. That is the test that
adding instrumentation did not change the computation.

## 3 · The decay

In [ ]:
df = logs["ch6 original (no KL)"]
cols = ["step", "reward_avg", "avg_response_len", "eval_acc"]
print(df[cols].iloc[::50].to_string(index=False))

peak = df.dropna(subset=["eval_acc"]).sort_values("eval_acc").iloc[-1]
print(f"\npeak accuracy {peak['eval_acc']}% at step {int(peak['step'])}")
print(f"final accuracy {df['eval_acc'].dropna().iloc[-1]}%")

**Predict first:** is `avg_response_len` correlated with `reward_avg`? Write your
guess down, then compute it.

The intuitive answer — "longer answers farm more reward" — is wrong here. It is
about **-0.74**: the long late-stage outputs are mostly wrong. That is the
difference between a policy that exploits the reward and one that has simply
degenerated.

In [ ]:
print("corr(reward, length) =",
      round(df["reward_avg"].corr(df["avg_response_len"]), 3))
print("length at step 1   :", round(df["avg_response_len"].iloc[0], 1))
print("length at step 500 :", round(df["avg_response_len"].iloc[-1], 1))
print("max length observed:", round(df["avg_response_len"].max(), 1))

## 4 · Prove the collapse

The KL run goes to 0.0% from step 100. Show that the KL term has taken over the
objective.

In [ ]:
kl = logs["+ KL term"]
kl["loss_minus_kl"] = (kl["loss"] - kl["kl_loss"]).abs()
print(kl[["step", "loss", "kl_loss", "loss_minus_kl", "entropy_avg", "eval_acc"]].tail(6).to_string(index=False))

**If `loss == kl_loss`, the policy-gradient term contributes nothing.** The
optimiser is only minimising divergence from the reference model.

In [ ]:
import math

VOCAB = 151_936
print("uniform entropy over the vocabulary: ln(V) =", round(math.log(VOCAB), 3))
print("entropy_avg at the end of the KL run      =",
      round(kl["entropy_avg"].dropna().iloc[-1], 3))
print("policy_ratio at the end                   =",
      round(kl["policy_ratio"].dropna().iloc[-1], 3))

**That is the punchline of the chapter.** `entropy_avg` ≈ ln(151,936) means the
model's output distribution is uniform over the entire vocabulary. It is not
"uncertain" — it has been trained to be maximally uncertain, because that is what
minimising KL against a reference rewards when the coefficient is too large.

The diagnostic is one line: **if `kl_loss` equals `loss`, stop the run.**

## 5 · Clipping, and what it bought

| Step | ch6 original | + clipped ratio |
| ---: | ---: | ---: |
| 50 | 46.2% | 45.2% |
| 250 | 41.2% | 41.6% |
| 500 | 33.0% | **41.6%** |

Clipping gives up a fraction of a point at the peak and keeps 8.6 points at the
end.

## 6 · Write the run report

Three sentences a colleague could act on, using only these CSVs. Draft it in the
cell below before reading on.

In [ ]:
# Your run report here, as a string.
report = """
Run: <which variant>. Peak accuracy <x>% at step <n>, final <y>%.
Primary symptom: <metric> went from <a> to <b>, which indicates <cause>.
Action: <the one change you would make>.
"""
print(report)

## 7 · Exercises

1. Find the exact step where `loss == kl_loss` in the KL run.
2. For the clipped-ratio run, is `policy_ratio` better behaved than the original?
   Plot or describe both.
3. Which variant would you ship, and what would you change next? Defend it in two
   sentences.